# Tutorial 13-2: Manually Calculating Self-Attention with BERT
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

---
## 1. Extracting Reality from the Transformer
In lecture, we defined the attention function as a mapping between a query and a set of key-value pairs. In this tutorial, we will:
1. Load a pre-trained BERT model.
2. Extract a real word embedding (hidden state).
3. Access the model's actual weight matrices ($W_q, W_k, W_v$) for a specific layer.
4. Manually calculate the attention weights using the Scaled Dot-Product formula.

$$Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

In [ ]:
import torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel

# Load pre-trained BERT
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name, output_attentions=True)
model.eval() # Set to evaluation mode

# Real Input Data
sentence = "The chef who made the tasty pizza was very happy."
inputs = tokenizer(sentence, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

with torch.no_grad():
    outputs = model(**inputs)
    # Get embeddings from the first layer (layer 0)
    # This is the 'X' in our formula
    embeddings = outputs.last_hidden_state

print(f"Extracted embeddings for {len(tokens)} tokens with dimension {embeddings.shape[-1]}.")

## 2. Manual Calculation using Model Weights
We will now dig into the `encoder.layer[0].attention.self` module to find the weights the model actually used during its forward pass.

In [ ]:
# Accessing actual BERT weights for Layer 0
self_attn = model.encoder.layer[0].attention.self

# BERT stores Q, K, V as linear layers. We extract their weight matrices.
Wq = self_attn.query.weight # Query weights
Wk = self_attn.key.weight   # Key weights
Wv = self_attn.value.weight # Value weights

with torch.no_grad():
    # Step 1: Project embeddings to Q, K, V space
    # Formula: q = Qx, k = Kx, v = Vx
    # Note: Torch linear layers use xW^T + b. For simplicity, we ignore bias here.
    Q = F.linear(embeddings, Wq)
    K = F.linear(embeddings, Wk)
    V = F.linear(embeddings, Wv)

    # Step 2: Compute pairwise similarities (Scaled Dot-Product)
    #
    d_k = Q.shape[-1]
    attention_scores = torch.matmul(Q, K.transpose(-1, -2)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

    # Step 3: Normalize with Softmax to get attention weights
    attention_weights = F.softmax(attention_scores, dim=-1)

print("Manual Attention Weights for first few tokens:\n", attention_weights[0, :5, :5])

## 3. High-Level Visualization with BertViz
Now that you've seen the raw math, we can use `bertviz` to see how these weights translate into linguistic focus.

In [ ]:
!python -m pip install bertviz
from bertviz import head_view

# Re-run inference to get attention tensors
with torch.no_grad():
    outputs = model(**inputs)
    attention = outputs.attentions

# Display the Head View
# Use this to see how 'Chef' attends to 'Pizza' or 'Was'
head_view(attention, tokens)